# Checkpoint 2 (Week 2): Reinforcement Learning with Q-learning

**Lab 4 — Projects in Machine Learning and AI**

**Environment chosen:** `FrozenLake-v1` with `is_slippery=False` (deterministic 4x4 grid).

**Why this environment?** It is small (16 states, 4 actions), easy to visualize as a grid with arrow policies, and fast enough to run many episodes in a 90-minute Colab session. Using `is_slippery=False` makes the dynamics deterministic, which lets us clearly see how exploration and the learning rate interact without the noise of stochastic transitions.

**This notebook implements:**
- Part A: Environment setup + random-policy baseline.
- Part B: Tabular Q-learning with an epsilon-greedy behavior policy.
- Part C: Random baseline vs. two Q-learning variants (fixed epsilon, decaying epsilon), with quantitative metrics, a learning-curve plot, and a policy visualization.
- Part D: Discussion of exploration, a failure mode, and how alpha / gamma / epsilon interact.

In [ ]:
# Install dependencies (safe to re-run in Colab)
!pip install -q gymnasium numpy matplotlib pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 6)

ENV_ID = 'FrozenLake-v1'
ENV_KWARGS = dict(is_slippery=False)

def make_env():
    return gym.make(ENV_ID, **ENV_KWARGS)

---
## Part A: Environment + random-policy baseline

The map is a 4x4 grid:
- `S` = start (top-left)
- `F` = frozen (safe)
- `H` = hole (episode ends with reward 0)
- `G` = goal (episode ends with reward +1)

Action encoding: `0 = Left`, `1 = Down`, `2 = Right`, `3 = Up`.

We start by running a uniform-random policy so we have a concrete baseline to beat.

In [ ]:
env = make_env()
n_states = env.observation_space.n
n_actions = env.action_space.n
print(f'States: {n_states}, Actions: {n_actions}')

desc = env.unwrapped.desc.astype(str)
print('Map:')
for row in desc:
    print(' '.join(row))
env.close()

In [ ]:
def run_random_policy(n_episodes=2000, max_steps=100, seed=1):
    """Uniform-random behavior policy baseline."""
    env = make_env()
    rng = np.random.default_rng(seed)
    episode_returns = []
    success_flags = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))
        ep_return = 0.0
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward
            steps += 1
        episode_returns.append(ep_return)
        success_flags.append(int(done and ep_return > 0))
    env.close()
    return np.array(episode_returns), np.array(success_flags)

rand_returns, rand_success = run_random_policy(n_episodes=2000)
print(f'Random baseline — avg return (last 100 eps): {rand_returns[-100:].mean():.3f}')
print(f'Random baseline — overall success rate:     {rand_success.mean():.3f}')

---
## Part B: Tabular Q-learning

We store the action-value function as a `(n_states, n_actions)` numpy array. During training we use an epsilon-greedy behavior policy. The Q-update is the standard off-policy temporal-difference update:

```
best_next = np.max(Q[next_state])
td_target = reward + gamma * best_next * (1 - done)
Q[state, action] += alpha * (td_target - Q[state, action])
```

Evaluation is separated from training: once Q is learned, we call `evaluate_policy` with epsilon = 0 (pure greedy).

In [ ]:
def train_q_learning(
    n_episodes=5000,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=1.0,
    epsilon_decay_episodes=None,
    max_steps=100,
    seed=42,
):
    """Tabular Q-learning.

    If epsilon_start == epsilon_end the epsilon is fixed.
    Otherwise epsilon is linearly interpolated from epsilon_start to epsilon_end
    over the first `epsilon_decay_episodes` episodes, then stays at epsilon_end.
    """
    env = make_env()
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))
    rng = np.random.default_rng(seed)

    episode_returns = []
    success_flags = []

    if epsilon_decay_episodes is None:
        epsilon_decay_episodes = n_episodes

    for ep in range(n_episodes):
        # Epsilon schedule
        if epsilon_start == epsilon_end:
            eps = epsilon_start
        else:
            frac = min(1.0, ep / max(1, epsilon_decay_episodes))
            eps = epsilon_start + frac * (epsilon_end - epsilon_start)

        state, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))
        ep_return = 0.0
        done = False
        steps = 0

        while not done and steps < max_steps:
            # Epsilon-greedy action selection
            if rng.random() < eps:
                action = int(rng.integers(0, n_actions))
            else:
                # Tie-break randomly to avoid biasing to action 0 when all Q are equal
                q_row = Q[state]
                max_q = q_row.max()
                best_actions = np.flatnonzero(q_row == max_q)
                action = int(rng.choice(best_actions))

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next = np.max(Q[next_state])
            td_target = reward + gamma * best_next * (1 - float(done))
            Q[state, action] = Q[state, action] + alpha * (td_target - Q[state, action])

            state = next_state
            ep_return += reward
            steps += 1

        episode_returns.append(ep_return)
        success_flags.append(int(done and ep_return > 0))

    env.close()
    return Q, np.array(episode_returns), np.array(success_flags)

In [ ]:
def evaluate_policy(Q, n_eval=100, max_steps=100, seed=7):
    """Evaluate the greedy policy derived from Q (epsilon = 0)."""
    env = make_env()
    rng = np.random.default_rng(seed)
    returns = []
    for ep in range(n_eval):
        state, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))
        ep_return = 0.0
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = int(np.argmax(Q[state]))
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward
            steps += 1
        returns.append(ep_return)
    env.close()
    returns = np.array(returns)
    return float(returns.mean()), float((returns > 0).mean())

---
## Part C: Required experiments

We compare three methods on the same environment:
1. **Random baseline** (uniform random actions).
2. **Q-learning with a fixed epsilon** of 0.1.
3. **Q-learning with a decaying epsilon** from 1.0 to 0.05 over the first 3000 episodes.

Everything else (alpha, gamma, number of episodes, seed) is held constant between the two Q-learning variants.

In [ ]:
N_EPISODES = 5000
ALPHA = 0.1
GAMMA = 0.99

Q_fixed, fixed_returns, fixed_success = train_q_learning(
    n_episodes=N_EPISODES,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_start=0.1,
    epsilon_end=0.1,
    seed=42,
)

Q_decay, decay_returns, decay_success = train_q_learning(
    n_episodes=N_EPISODES,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_episodes=3000,
    seed=42,
)

print('Training done.')
print(f'Fixed   ε=0.1  — final-100-ep training avg return: {fixed_returns[-100:].mean():.3f}')
print(f'Decay   1→0.05 — final-100-ep training avg return: {decay_returns[-100:].mean():.3f}')

In [ ]:
# Quantitative outcomes
def first_threshold_episode(returns, threshold=0.9, window=100):
    """First episode index where the rolling mean of the last `window` returns >= threshold.
    Returns None if it never happens.
    """
    if len(returns) < window:
        return None
    roll = np.convolve(returns, np.ones(window) / window, mode='valid')
    idx = np.where(roll >= threshold)[0]
    if len(idx) == 0:
        return None
    return int(idx[0] + window - 1)

eval_fixed_mean, eval_fixed_success = evaluate_policy(Q_fixed, n_eval=100)
eval_decay_mean, eval_decay_success = evaluate_policy(Q_decay, n_eval=100)

# For the random baseline the 'evaluation' success rate is just the empirical success rate
rand_eval_success = float(rand_success[-100:].mean())

results = pd.DataFrame([
    {
        'method': 'Random baseline',
        'avg_return_last_100': float(rand_returns[-100:].mean()),
        'greedy_eval_success_rate': rand_eval_success,
        'episodes_to_0.9_threshold': first_threshold_episode(rand_returns),
    },
    {
        'method': 'Q-learning (fixed ε=0.1)',
        'avg_return_last_100': float(fixed_returns[-100:].mean()),
        'greedy_eval_success_rate': eval_fixed_success,
        'episodes_to_0.9_threshold': first_threshold_episode(fixed_returns),
    },
    {
        'method': 'Q-learning (decaying ε 1.0→0.05)',
        'avg_return_last_100': float(decay_returns[-100:].mean()),
        'greedy_eval_success_rate': eval_decay_success,
        'episodes_to_0.9_threshold': first_threshold_episode(decay_returns),
    },
])

results

In [ ]:
# Learning-curve figure (smoothed with a 100-episode moving average)
def smooth(x, window=100):
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode='valid')

plt.figure(figsize=(10, 6))
plt.plot(smooth(rand_returns), label='Random baseline', alpha=0.7)
plt.plot(smooth(fixed_returns), label='Q-learning (fixed ε=0.1)', alpha=0.9)
plt.plot(smooth(decay_returns), label='Q-learning (decaying ε 1.0→0.05)', alpha=0.9)
plt.xlabel('Episode (rolling 100-episode mean)')
plt.ylabel('Average episode return')
plt.title(f'Learning curves on {ENV_ID} (is_slippery=False)')
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Final-policy visualization: arrows + state-value heatmap + text table
ARROWS = {0: '←', 1: '↓', 2: '→', 3: '↑'}

def visualize_policy(Q, title):
    env = make_env()
    desc = env.unwrapped.desc.astype(str)
    env.close()
    nrows, ncols = desc.shape
    policy = np.argmax(Q, axis=1).reshape(nrows, ncols)
    V = np.max(Q, axis=1).reshape(nrows, ncols)

    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(V, cmap='viridis')
    for r in range(nrows):
        for c in range(ncols):
            cell = desc[r, c]
            if cell == 'H':
                ax.text(c, r, 'H', ha='center', va='center',
                        color='red', fontsize=22, fontweight='bold')
            elif cell == 'G':
                ax.text(c, r, 'G', ha='center', va='center',
                        color='gold', fontsize=22, fontweight='bold')
            else:
                prefix = 'S\n' if cell == 'S' else ''
                ax.text(c, r, f'{prefix}{ARROWS[int(policy[r, c])]}',
                        ha='center', va='center', color='white', fontsize=20)
    ax.set_xticks(range(ncols))
    ax.set_yticks(range(nrows))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='max_a Q(s,a)')
    plt.show()

    # Also print a readable text table
    print(f'Policy table — {title}:')
    for r in range(nrows):
        row_cells = []
        for c in range(ncols):
            cell = desc[r, c]
            if cell == 'H':
                row_cells.append(' H ')
            elif cell == 'G':
                row_cells.append(' G ')
            else:
                row_cells.append(f' {ARROWS[int(policy[r, c])]} ')
        print(''.join(row_cells))
    print()

visualize_policy(Q_fixed, 'Final policy — Q-learning (fixed ε=0.1)')
visualize_policy(Q_decay, 'Final policy — Q-learning (decaying ε 1.0→0.05)')

---
## Part D: Interpretation

### How exploration affected learning

The learning curve above shows something that is at first a little counter-intuitive: the **fixed ε=0.1** run climbs to ~0.9 average return almost immediately (within roughly the first 100 episodes), while the **decaying ε 1.0→0.05** run climbs slowly and does not catch up until around episode 3000. On a textbook sparse-reward problem you would expect the opposite, so it is worth being precise about why this happened on *this* environment.

Two things conspire to make the fixed-ε variant very effective here:

1. **`is_slippery=False` is deterministic, and the optimal path is short (six steps from S to G).** That means once the agent stumbles into the goal *once*, the TD update immediately starts propagating a useful signal backwards along that exact trajectory, and the next time the agent revisits any of those states it has a real best action to exploit.
2. **Our ε-greedy implementation tie-breaks argmax uniformly at random when Q-values are equal** (see the `np.flatnonzero(q_row == max_q)` line in `train_q_learning`). So when the Q-table is initialized to zero, the 'greedy' branch of the fixed-ε=0.1 policy is effectively a uniform random action anyway. The fixed-ε agent therefore explores *just as well* as a purely random agent at the start, and then, the moment any Q-value becomes non-zero, it switches instantly to exploiting it 90% of the time.

The decaying-ε agent, by contrast, *keeps* acting nearly uniformly at random long after it has already discovered the goal. At episode 500 it still has ε ≈ 0.83, so even when Q(s, a) already points at the correct action, the agent only follows it ~17% of the time. It is wasting most of its updates on states it will never visit under the optimal policy. Only once ε drops below ~0.1 (around episode 2700 with our linear schedule) does the effective policy become 'mostly greedy,' and that is exactly where the green curve finally catches up to the orange one in the plot.

So the reading of this figure is: **on a small, deterministic environment with a short optimal path, aggressive early exploration is unnecessary, and a decaying schedule that starts too high is actually a handicap.** We would expect the decaying schedule to *win* on a larger or noisier environment — e.g. slippery FrozenLake or a bigger map — where the agent genuinely needs to try many trajectories before it finds one that reaches the goal, and where a fixed ε=0.1 might never cover enough of the state space to find the reward at all.

### One failure mode we observed

**Over-exploration: a decay schedule that starts too high and decays too slowly wastes training.** The green curve in the learning-curve figure is the concrete failure case: the agent has all the information it needs to behave well by roughly episode 500–1000, but because ε is still in the 0.7–0.8 range it keeps throwing dice and does not get to cash in that knowledge until the schedule has burned down. The symptom is a learning curve that climbs in a long, roughly linear ramp instead of the sharp elbow you see in the fixed-ε run. Possible fixes: (a) start the decay lower (e.g. 0.5 → 0.05), (b) decay exponentially rather than linearly, or (c) decay over fewer episodes — the right number is 'just long enough to find the goal a handful of times,' which on a 4x4 deterministic grid is a few hundred episodes, not 3000. We deliberately left the bad schedule in so the plot shows the effect cleanly.

Worth noting: the *other* classic failure mode — slow convergence under a fixed low ε because the agent never discovers the goal — would appear on slippery FrozenLake or on a bigger map, but it does not bite us here for the two reasons in the previous section (deterministic short path + random tie-break on Q=0).

### How alpha, gamma, and epsilon interact in this environment

- **alpha (learning rate = 0.1):** how much each TD error moves `Q[s, a]`. Because transitions are deterministic, the TD target `r + γ·max Q[s']` is noise-free, so alpha can be fairly large (0.1–0.5) without oscillation. Too small and the first successful trajectory barely registers; too large would not hurt on this env but would cause ringing under stochastic transitions.
- **gamma (discount = 0.99):** how far the +1 at the goal propagates backward. With γ=0.99 the start state still sees a value of ≈ 0.99⁶ ≈ 0.94 after the reward has been back-propagated along the six-step optimal path, which is plenty to induce a clean greedy policy from the very first state. If we lowered γ to 0.5, the start state would see only 0.5⁶ ≈ 0.016, which is within numerical noise of the hole states, and the greedy policy near S would become essentially arbitrary.
- **epsilon (exploration rate):** the 'how much does the behavior policy deviate from the current best guess' knob. On this environment the empirically best setting is 'low and constant, with random tie-breaking on ties' — which is what the fixed-ε=0.1 run does — because the random tie-break provides all the exploration you need until Q learns something real.
- **Interaction:** the three knobs form a pipeline. Epsilon decides *which* (s, a) pairs get visited; alpha decides *how fast* each visit moves Q; gamma decides *how far* each visited update's information can travel. On a deterministic short-horizon env, γ and α can be set aggressively because the targets are clean, so the only remaining question is whether ε is high enough to visit the states that matter — and here, 'just use the argmax with a random tie-break and a little noise' already does that. On a harder environment the balance shifts: you need more ε (to visit enough states), a smaller α (because noisy targets would otherwise push Q around), and roughly the same γ.

---
## Submission checklist

- [x] A working Q-table implementation (`train_q_learning`, Part B).
- [x] A random baseline (`run_random_policy`, Part A) and two Q-learning variants (fixed ε, decaying ε, Part C).
- [x] Quantitative outcomes: `avg_return_last_100`, `greedy_eval_success_rate`, and `episodes_to_0.9_threshold` in the results table.
- [x] A learning-curve figure comparing all three methods.
- [x] A final-policy visualization (arrows on a value heatmap + text table) for both Q-learning variants.
- [x] Discussion of exploration, a failure mode, and alpha/gamma/epsilon interaction (Part D).